## Pandas - Group By (raggruppamenti)

In [22]:
import pandas as pd

In [23]:
data = {
    'codice':range(1001, 1006),
    'nome':['mela', 'pera', 'kiwi', 'banana', 'fragola'],
    'qta':[12,23,34,45,56],
    'id_magazzino':[1,2,2,1,1] 
}
dfp = pd.DataFrame(data)
dfp

,codice,nome,qta,id_magazzino
0,1001,mela,12,1
1,1002,pera,23,2
2,1003,kiwi,34,2
3,1004,banana,45,1
4,1005,fragola,56,1


In [24]:
dfp.groupby("id_magazzino").groups

{1: [0, 3, 4], 2: [1, 2]}

In [25]:
dfp.groupby("id_magazzino").get_group(1)

,codice,nome,qta,id_magazzino
0,1001,mela,12,1
3,1004,banana,45,1
4,1005,fragola,56,1


In [26]:
dfp.groupby("id_magazzino").get_group(2)

,codice,nome,qta,id_magazzino
1,1002,pera,23,2
2,1003,kiwi,34,2


In [27]:
dfp["id_magazzino"].unique()

array([1, 2], dtype=int64)

In [28]:
dfp.groupby("id_magazzino")['qta'].sum()

id_magazzino
1    113
2     57
Name: qta, dtype: int64

In [29]:
dfp.groupby("id_magazzino")['qta'].describe()

,count,mean,std,min,25%,50%,75%,max
id_magazzino,,,,,,,,
1,3.0,37.666667,22.898326,12.0,28.50,45.0,50.50,56.0
2,2.0,28.500000,7.778175,23.0,25.75,28.5,31.25,34.0


### Pandas e Excel

Per leggere dati da file excel uso la libreria openpyxl

In [30]:
#%pip install openpyxl
# import pandas as pd
df1 = pd.read_excel("persone.xlsx")
df1

,nome,cognome,data_nascita,peso,altezza
0,Mario,Rossi,01/01/1980,75,178
1,Maria,Verdi,15/03/1992,60,165
2,Giovanni,Blu,22/07/1975,80,182
3,Anna,Neri,05/11/1998,55,160
4,Luca,Bianchi,18/04/1963,90,175
5,Sara,Gialli,29/12/2000,48,158
6,Davide,Viola,10/08/1987,72,180
7,Elena,Rosa,03/02/1978,65,168
8,Marco,Marroni,17/06/1995,78,183
9,Beatrice,Azzurri,24/09/2002,52,155


**Scrittura Excel**: servirà la libreria xlswriter

%pip install xlswriter

In [31]:
#genero dei movimenti su un conto bancario
import numpy as np

#creo una lista di mesi
mesi = ["ge", 'fe', 'ma', 'ap', 'mg', 'gi', 'lu', 'ag', 'st', 'ot', 'nv', 'di']

#genero dei valori casuali per simulare i movimenti in ingresso/uscita del mio cc
ingressi = np.random.randint(1000, 5000, size=12)
uscite = np.random.randint(1000, 5000, size=12)

#combino i dati in un dataframe
cc = pd.DataFrame( {"mese":mesi, "entrate":ingressi, "uscite":uscite})
cc

,mese,entrate,uscite
0,ge,2645,1222
1,fe,1083,1050
2,ma,4488,1957
3,ap,2063,3978
4,mg,3864,1837
5,gi,3390,1347
6,lu,3408,4420
7,ag,4676,3081
8,st,2752,2419
9,ot,2088,4394


In [32]:
wr = pd.ExcelWriter("conto.xlsx", engine="xlsxwriter")
cc.to_excel(wr, index=False, sheet_name="miocc")
wr.close()

In [33]:
with pd.ExcelWriter("conto.xlsx", engine="xlsxwriter") as wr:
    cc.to_excel(wr, index=False, sheet_name="miocc")

In [34]:
#esempio con modifica celle del file excel
with pd.ExcelWriter("conto.xlsx", engine="xlsxwriter") as wr:
    cc.to_excel(wr, index=False, sheet_name="miocc")
    workbook = wr.book
    #seleziono il tab su cui lavorare
    ws = wr.sheets["miocc"]
    f1 = workbook.add_format({'bold':True, 'font_color':"blue"})
    ws.write("D2", "Hello", f1)


### Pandas e Database
Usando la libreria sqlalchemy. Installo con:

%pip install sqlalchemy

In [35]:
from sqlalchemy import create_engine

#creo una connessione al database
eng = create_engine("sqlite:///banca.db")
# trasferisco il dataframe in una tabella
cc.to_sql("conto", eng, index=False)


12

In [36]:
#dalla tabella del database al dataframe
with eng.connect() as c:
    df2 = pd.read_sql_table("conto", c)

df2.head(3)

,mese,entrate,uscite
0,ge,2645,1222
1,fe,1083,1050
2,ma,4488,1957


In [37]:
with eng.connect() as c:
    df3 = pd.read_sql_query("select * from conto where entrate > 2500 order by entrate", c)
df3

,mese,entrate,uscite
0,ge,2645,1222
1,st,2752,2419
2,gi,3390,1347
3,lu,3408,4420
4,di,3753,1596
5,mg,3864,1837
6,ma,4488,1957
7,ag,4676,3081


In [38]:
#chiudo la connessione al file
eng.dispose()

In [39]:
#rimovo il database
#import os
#os.remove("banca.db")